# Fig. 7e: ground truth subjected to the undersampling and the selection bias of Kilosort 4 (evoked model)

**author**: laquitainesteeve@gmail.com

**purpose**: reproduce panel e of the paper: the 3D manifold of a sample of the ground truth neurons that matches both the yield of Kilosort 4 (354) and its distribution of unit types: the undersampling and the selection bias of Kilosort 4. Source data and svg are saved in figures/8_source_data/fig7e/.

**method (as in the paper's notebook fig7_A_pca_manifolds_revised.ipynb)**
* same data and analysis as fig7c.ipynb (1-hour evoked simulation, 360 directions x 50 repeated trials, spike counts in the 200 ms stimulus window, PCA with 3 components of the trials, colour = stimulus direction), applied to a sample of the ground truth neurons
* Kilosort 4 has 354 sorted single-units (units of `sorting_quality_evoked_1h.csv`); the type of a sorted unit is its layer (from the quality file), and the synapse class and etype of its best matching ground truth neuron (agreement score, delta_time = 1.3 ms)
* sample: for each type of the sorted single-units, as many ground truth neurons of that type as there are sorted units of the type (drawn without replacement, with replacement if the ground truth has fewer neurons of the type; seed 0, as in the paper's notebook)

In [1]:
%load_ext autoreload
%autoreload 2
import os
import numpy as np
import pandas as pd
import spikeinterface as si
from matplotlib import pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# set project path
PROJ_PATH = "/home/steeve/steeve/epfl/code/spikebias/"
os.chdir(PROJ_PATH)

# custom package
from sklearn.decomposition import PCA
from src.nodes.task import task
import random
from src.nodes.analysis.features import features as feat

# paths
GT_PATH = "dataset/00_raw/ground_truth_npx_evoked"
K4_PATH = "dataset/01_intermediate/sorting/npx_evoked/SortingKS4"       # Kilosort 4, 1 hour
QUALITY_PATH = "dataset/01_intermediate/analysis/sorting_quality/sorting_quality_evoked_1h.csv"     # sorted single-units
DT = 1.3     # ms
SEED_RANDOM_K4_YIELD = 6        # seeds of the paper's notebook
SEED_BIASED_AS_K4 = 0

# stimulus: 360 directions x 50 repeated trials
SAMPLE_DIR = np.arange(0, 360, 1)

# FIGURE SETTINGS
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 6  # 5-7 with Nature neuroscience as reference
plt.rcParams["lines.linewidth"] = 0.5
plt.rcParams["axes.linewidth"] = 0.5
plt.rcParams["axes.spines.top"] = False
plt.rcParams["xtick.major.width"] = 0.5
plt.rcParams["xtick.minor.width"] = 0.5
plt.rcParams["ytick.major.width"] = 0.5
plt.rcParams["ytick.minor.width"] = 0.5
plt.rcParams["xtick.major.size"] = 3.5 * 1.1
plt.rcParams["xtick.minor.size"] = 2 * 1.1
plt.rcParams["ytick.major.size"] = 3.5 * 1.1
plt.rcParams["ytick.minor.size"] = 2 * 1.1
legend_cfg = {"frameon": False, "handletextpad": 0.5}

# save config
savefig_cfg = {"transparent": True, "dpi": 300}

In [ ]:
# Nature figure guidelines for every saved figure: Arial, 5-7 pt, editable text, images of at least 300 dpi, a .pdf next to each .svg
import os, sys
sys.path.insert(0, os.getcwd())     # the repository (the notebook moved there above)
from src.nodes import figure_style
figure_style.apply()

In [2]:
%%time

# trials of the 360 directions and their direction labels
trial = task.get_direction_trials(SAMPLE_DIR)
direction = task.get_stimulus_labels()[trial].astype(float)

# task epochs (ms)
task_pm = task.get_task_parameters(start_delay=500, n_orientations=10, n_repeats=50, stimulus_duration=200, n_simulations=36)
stimulus_intervals_ms = task.get_stimulus_intervals_ms(task_pm["epoch_labels"], task_pm["epoch_ms"])

# neurons x trials spike counts
SortingTrue = si.load_extractor(GT_PATH)
response = task.compute_response_by_stim_matrix(SortingTrue.unit_ids, SortingTrue, stimulus_intervals_ms)[:, trial]
print("ground truth neurons:", response.shape[0], "| trials:", response.shape[1])

ground truth neurons: 1836 | trials: 18000
CPU times: user 14 s, sys: 107 ms, total: 14.1 s
Wall time: 14.1 s


## 3D PCA manifold

## Sample the ground truth neurons

In [3]:
# --- sorted single-units of Kilosort 4: types (layer from the quality file; synapse class and etype of the best matching ground truth neuron) ---
Sorting = si.load_extractor(K4_PATH)
single_units = pd.read_csv(QUALITY_PATH).query("sorter == 'KS4'").set_index("sorted")
matched = feat.get_sorted_unit_features(Sorting, SortingTrue, DT)
matched.index = matched.index.astype(int)                 # the unit ids of the sorting are strings
df_sort = single_units[["layer"]].join(matched[["synapse", "etype"]])
assert df_sort.notna().all().all(), "all sorted single-units should have a type"
TYPE_FEAT = ["layer", "synapse", "etype"]
df_sort = df_sort[TYPE_FEAT]
df_gt = feat.get_unit_features(SortingTrue)[TYPE_FEAT]
n_single_units = len(df_sort)
print("Kilosort 4 single-units:", n_single_units, "| ground truth neurons:", len(df_gt))

def sample_units_randomly_to_match_sorter_yield(n_gt, n_units, seed):
    """random sample of ground truth neurons (undersampling) with as many neurons as the sorter has single-units"""
    random.seed(seed)      # use random.seed with random.sample and not np.random.seed
    return random.sample(np.arange(0, n_gt, 1).tolist(), n_units)

def locate_units_of_type(this_type, df_unit, type_count, seed):
    """locations of `type_count` ground truth neurons of the type `this_type`, sampled without replacement (with replacement if the ground truth has fewer)"""
    flagged = df_unit.T.apply(lambda row: all(row.values == this_type.values)).values
    type_loc = np.where(flagged)[0]
    random.seed(seed)
    try:
        ix = random.sample(np.arange(0, len(type_loc), 1).tolist(), type_count)
    except ValueError:
        ix = random.choices(np.arange(0, len(type_loc), 1).tolist(), k=type_count)
    return type_loc[ix].tolist()

def sample_units_to_match_sorter_sampling_bias(df_gt, df_sort, seed):
    """sample ground truth neurons to match the sorter's yield and its distribution of unit types: for each type, as many ground truth neurons as sorted units"""
    unique_type_sort = df_sort.drop_duplicates()
    counts_sort = feat.count_unit_type(df_sort, unique_type_sort)
    gt_loc = []
    for ix in range(len(unique_type_sort)):
        assert tuple(counts_sort.iloc[ix]["type"]) == tuple(unique_type_sort.iloc[ix]), "features should be the same"
        gt_loc += locate_units_of_type(unique_type_sort.iloc[ix], df_gt, counts_sort.iloc[ix]["count"], seed)
    return gt_loc

sampled = sample_units_to_match_sorter_sampling_bias(df_gt, df_sort, SEED_BIASED_AS_K4)

Kilosort 4 single-units: 354 | ground truth neurons: 1836


## 3D PCA manifold

In [4]:
%%time

manifold = PCA(n_components=3, random_state=0).fit_transform(response[sampled].T)
manifold = manifold / np.max(np.abs(manifold))
df = pd.DataFrame({"trial": np.arange(len(direction)), "direction_deg": direction, "latent_1": manifold[:, 0], "latent_2": manifold[:, 1], "latent_3": manifold[:, 2]})
df_units = pd.DataFrame({"ground_truth_unit_id": SortingTrue.unit_ids[sampled], "layer": df_gt.iloc[sampled]["layer"].values, "synapse_class": df_gt.iloc[sampled]["synapse"].values, "etype": df_gt.iloc[sampled]["etype"].values})
print("neurons in the population:", response[sampled].shape[0], "| distinct neurons:", df_units["ground_truth_unit_id"].nunique())
display(df.head(3))

neurons in the population: 354 | distinct neurons: 354


,trial,direction_deg,latent_1,latent_2,latent_3
0,0,0.0,0.002817,0.450573,0.444112
1,1,0.0,-0.227996,0.089882,0.008981
2,2,0.0,-0.231743,0.063540,0.210479


CPU times: user 133 ms, sys: 10 ms, total: 143 ms
Wall time: 143 ms


## Plot, same as the paper's panel fig7e

In [5]:
# paper geometry (page 22 of the paper): canvas 59.6 x 75.8 pt, Arial 5 pt text, editable (src/nodes/validation/manifold_plots.py)
from src.nodes.validation.manifold_plots import plot_single_manifold, SELECTED_PANEL

def plot_source_data(df: pd.DataFrame):
    return plot_single_manifold(df, SELECTED_PANEL)

## Plot and save the figure and the source data

In [6]:
%%time

# plot
fig = plot_source_data(df)

# save figure
os.makedirs("figures/8_source_data/fig7e", exist_ok=True)
plt.savefig("figures/8_source_data/fig7e/fig7e.svg", transparent=True)   # the .pdf is written next to the .svg

# save source data (sheet 1: 3D manifold of the trials; sheet 2: ground truth neurons of the population)
with pd.ExcelWriter("figures/8_source_data/fig7e/fig7e.xlsx") as writer:
    df.to_excel(writer, sheet_name="manifold_3d_trials", index=False)
    df_units.to_excel(writer, sheet_name="neurons", index=False)

## Check: replot from the saved source data

In [7]:
%%time

df = pd.read_excel("figures/8_source_data/fig7e/fig7e.xlsx", sheet_name="manifold_3d_trials")
fig = plot_source_data(df)